# 03 Raw to Bronze

## Purpose

This notebook runs the Raw → Bronze transformation for the OpenAQ warehouse.

Raw contains immutable OpenAQ API responses together with ingestion lineage. Bronze is the first schema-controlled Delta layer.

The transformation logic now lives in the packaged module `spark/transformations/bronze.py` (`build_bronze`), with the explicit Raw contract schemas in `spark/schemas/bronze_contract.py`. This notebook is a thin orchestration layer: it configures the environment and calls the packaged write path, which covers source validation, the Raw contract gate, normalization, standardization, timestamp-parse validation, Delta persistence, and persisted-vs-source reconciliation for measurements and the countries/locations/sensors/parameters metadata entities.

Raw data is never modified by this notebook.


## Setup

Configure the Raw source location and Bronze destination, and import the packaged transformation. One-off exploration cells from the original development notebook (schema-inference reviews, format-identification displays) moved to the module or were dropped.

In [ ]:
CATALOG = "workspace"
RAW_ROOT = "/Volumes/workspace/raw/openaq_landing/"

import sys
PACKAGE_ROOT = "/Workspace/Users/dammifabz@gmail.com/.bundle/openaq-databricks/dev/files"  # bundle-deployed spark/ package; adjust for exploration
if PACKAGE_ROOT not in sys.path:
    sys.path.append(PACKAGE_ROOT)

from spark.transformations.bronze import build_bronze


## Run Raw → Bronze

Execute the packaged write path: validate sources, enforce the Raw measurement contract, standardize Bronze columns, persist managed Delta tables under `workspace.bronze`, reconcile persisted vs source counts, and check the required parameter IDs.

In [ ]:
counts = build_bronze(spark, RAW_ROOT, CATALOG)

for table_name, persisted_count in counts.items():
    print(f"{table_name}: {persisted_count:,} rows persisted")

display(counts)
